# 09 — Ablation 8 đặc trưng lịch sử trên seed 43

So sánh công bằng 5 cấu hình: **A**, **H-R**, **H-F**, **H-M**, **H-RFM**. Mọi cấu hình giữ nguyên FraudGT, sampling và huấn luyện; chỉ thay nhóm đặc trưng lịch sử.

Giao thức chính: threshold cố định **0.50**; best epoch được chọn bằng **validation F1**, sau đó đọc test metric tại đúng epoch đó. Test không tham gia chọn epoch.

In [ ]:
SEED = 43
THRESHOLD = 0.50
NUM_THREADS = 2
NUM_WORKERS = 2
MODELS = ['A', 'H-R', 'H-F', 'H-M', 'H-RFM']
print('Seed:', SEED, '| threshold:', THRESHOLD)
print('Models:', MODELS)

## 1. Kiểm tra môi trường

In [ ]:
import platform, subprocess, sys, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__, '| CUDA runtime:', torch.version.cuda)
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)
if torch.cuda.device_count() == 0:
    raise RuntimeError('Notebook này cần GPU.')

## 2. Cài dependency

In [ ]:
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_url)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'torchmetrics', 'yacs', 'datatable',
                'pandas', 'matplotlib', 'wandb', 'ogb', 'tensorboardX',
                'pyyaml'], check=True)
print('Dependencies installed.')

## 3. Lấy đúng mã nguồn và ghi commit

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/mhiunguyen/TH-FraudGT.git'
repo = Path('/kaggle/working/TH-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', repo)
print('Commit:', commit)
required = [repo / 'scripts/run_history_ablation.py',
            repo / 'scripts/summarize_history_ablation.py',
            repo / 'configs/AML-Small-HI/AML-Small-HI-Ablation-H-RFM-Seed43.yaml']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError('Repository chưa có bản ablation mới: ' + str(missing))

## 4. Gắn dữ liệu IBM AML Small-HI

In [ ]:
from shutil import copy2

candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError('Hãy Add Input bộ IBM AML; không tìm thấy HI-Small_Trans.csv.')
source = candidates[0]
destination = repo / 'data' / 'AML' / 'HI-Small_Trans.csv'
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != source.stat().st_size:
    copy2(source, destination)
print('Dataset:', destination)
print(f'Size: {destination.stat().st_size / 1024**2:.1f} MiB')

## 5. Kiểm tra cấu hình công bằng và ánh xạ R/F/M

In [ ]:
import copy, yaml
sys.path.insert(0, str(repo))
from fraudGT.datasets.history_features import (
    HISTORY_FEATURE_NAMES, history_feature_names, resolve_history_feature_indices)

cfg_root = repo / 'configs' / 'AML-Small-HI'
configs = {}
for model in MODELS:
    path = cfg_root / f'AML-Small-HI-Ablation-{model}-Seed43.yaml'
    cfg = yaml.safe_load(path.read_text(encoding='utf-8'))
    assert Path(cfg['out_dir']) == repo / 'results'
    assert Path(cfg['dataset']['dir']) == repo / 'data'
    assert cfg['seed'] == SEED
    assert cfg['num_threads'] == NUM_THREADS
    assert cfg['num_workers'] == NUM_WORKERS
    configs[model] = cfg

reference = copy.deepcopy(configs['A'])
reference['dataset'].pop('add_history')
reference['dataset'].pop('history_groups')
for model in MODELS[1:]:
    candidate = copy.deepcopy(configs[model])
    candidate['dataset'].pop('add_history')
    candidate['dataset'].pop('history_groups')
    assert candidate == reference, f'{model} khác A ngoài cấu hình history'

for model in MODELS:
    groups = configs[model]['dataset']['history_groups']
    selected = [] if model == 'A' else list(history_feature_names(groups))
    add_history = configs[model]['dataset']['add_history']
    print(f'{model:5s}: add_history={add_history}; ' +
          f'groups={groups}; features={len(selected)}')
    for name in selected:
        print('       -', name)
assert resolve_history_feature_indices(['recency']) == (0, 1, 2, 3)
assert resolve_history_feature_indices(['frequency']) == (4, 5, 6)
assert resolve_history_feature_indices(['monetary']) == (7,)
print('PASS: năm cấu hình chỉ khác nhóm đặc trưng lịch sử.')

## 6. Kiểm tra chống rò rỉ trên dữ liệu đồ chơi

In [ ]:
import numpy as np, pandas as pd
from fraudGT.datasets.history_features import compute_past_only_history_features_raw

toy = pd.DataFrame({
    'from_id': [0, 0, 0, 3],
    'to_id': [1, 2, 1, 1],
    'Timestamp': [10, 10, 20, 20],
    'Amount Received': [10.0, 20.0, 30.0, 5.0],
})
toy_history = compute_past_only_history_features_raw(toy)
display(pd.concat([toy, pd.DataFrame(toy_history, columns=HISTORY_FEATURE_NAMES)], axis=1))
np.testing.assert_allclose(toy_history[0], np.zeros(8), atol=1e-7)
np.testing.assert_allclose(toy_history[1], np.zeros(8), atol=1e-7)
assert np.isclose(toy_history[2, 4], np.log1p(2))
assert np.isclose(toy_history[2, 6], np.log1p(1))
assert np.isclose(toy_history[3, 5], np.log1p(1))
print('PASS: giao dịch cùng timestamp không nhìn thấy nhau; chỉ dùng t < hiện tại.')

## 7. Tạo cache A và cache lịch sử tuần tự

Bước này tránh hai tiến trình ghi cache đồng thời. Cache lịch sử luôn chứa đủ tám đặc trưng; H-R/H-F/H-M chỉ chọn đúng cột trong bộ nhớ nên không phải tiền xử lý lại.

In [ ]:
import gc, time
from fraudGT.datasets.aml_dataset import AMLDataset

cache_specs = [
    ('A', False, ['recency', 'frequency', 'monetary'], 5),
    ('H-RFM', True, ['recency', 'frequency', 'monetary'], 13),
]
for name, add_history, groups, expected_dim_with_port in cache_specs:
    started = time.time()
    dataset = AMLDataset(root=str(repo / 'data' / 'AML'), name='Small-HI',
                         reverse_mp=True, add_ports=True,
                         add_history=add_history, history_groups=groups)
    dim = dataset.data_dict['train']['node', 'to', 'node'].edge_attr.shape[1]
    assert dim == expected_dim_with_port, (name, dim, expected_dim_with_port)
    print(name, '| edge dim:', dim, '| cache:', dataset.processed_paths)
    print(f'{name} ready in {(time.time() - started) / 60:.1f} minutes')
    del dataset
    gc.collect()

for name, groups, expected_dim in [
    ('H-R', ['recency'], 9), ('H-F', ['frequency'], 8), ('H-M', ['monetary'], 6)]:
    dataset = AMLDataset(root=str(repo / 'data' / 'AML'), name='Small-HI',
                         reverse_mp=True, add_ports=True, add_history=True,
                         history_groups=groups)
    dim = dataset.data_dict['train']['node', 'to', 'node'].edge_attr.shape[1]
    assert dim == expected_dim, (name, dim, expected_dim)
    print(name, '| verified edge dim:', dim)
    del dataset
    gc.collect()
print('PASS: kích thước A/R/F/M/RFM lần lượt là 5/9/8/6/13 (đã gồm Port).')

## 8. Huấn luyện năm cấu hình

Trên 2×T4, notebook chạy hai job song song theo từng pha. Dự kiến khoảng 80–100 phút sau khi cache đã sẵn sàng. Nếu cell bị chạy lại, kết quả hoàn tất được tự động bỏ qua.

In [ ]:
runner = repo / 'scripts' / 'run_history_ablation.py'
gpus = ['0', '1'] if torch.cuda.device_count() >= 2 else ['0']
cmd = [sys.executable, '-u', str(runner), '--repo', str(repo), '--gpus'] + gpus
print('Command:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 9. Tổng hợp đúng giao thức

In [ ]:
SUMMARY = Path('/kaggle/working/summary_history_ablation_seed43_fixed050.csv')
summarizer = repo / 'scripts' / 'summarize_history_ablation.py'
cmd = [sys.executable, str(summarizer), '--results-root', str(repo / 'results'),
       '--output', str(SUMMARY), '--threshold', str(THRESHOLD)]
subprocess.run(cmd, check=True)
results = pd.read_csv(SUMMARY)
display(results[['model', 'best_epoch_by_validation', 'threshold', 'val_f1',
                 'test_f1', 'delta_f1_vs_A', 'test_precision', 'test_recall',
                 'test_auc', 'parameters', 'gpu_memory_mib']])
print('Lưu ý: đây là ablation seed 43 theo hướng dẫn hiện tại, chưa phải kết luận nhiều seed.')

## 10. Biểu đồ đơn giản

In [ ]:
import matplotlib.pyplot as plt

plot_data = results.set_index('model').loc[MODELS].reset_index()
fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(plot_data['model'], plot_data['test_f1'] * 100,
              color=['#666666', '#8ca6c0', '#6f91b3', '#547da5', '#315f8d'])
baseline = float(plot_data.loc[plot_data['model'] == 'A', 'test_f1'].iloc[0]) * 100
ax.axhline(baseline, color='black', linestyle='--', linewidth=1,
           label=f'FraudGT gốc: {baseline:.2f}%')
for bar, value in zip(bars, plot_data['test_f1'] * 100):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.5, f'{value:.2f}%',
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('F1 trên tập kiểm thử (%)')
ax.set_xlabel('Cấu hình')
ax.set_title('Ảnh hưởng của từng nhóm đặc trưng lịch sử — seed 43')
ax.grid(axis='y', alpha=0.2)
ax.legend()
fig.tight_layout()
PLOT = Path('/kaggle/working/history_ablation_seed43_f1.png')
fig.savefig(PLOT, dpi=180, bbox_inches='tight')
plt.show()
print('Plot:', PLOT)

## 11. Đóng gói bằng chứng để tải về

In [ ]:
import shutil

bundle = Path('/kaggle/working/H_history_ablation_seed43_artifacts')
bundle.mkdir(exist_ok=True)
files = [SUMMARY, PLOT, repo / 'fraudGT/datasets/history_features.py',
         repo / 'tests/test_history_features.py']
files += [cfg_root / f'AML-Small-HI-Ablation-{model}-Seed43.yaml' for model in MODELS]
files += [Path('/kaggle/working') / f'ablation_{model}_seed43.log' for model in MODELS]
for path in files:
    if path.exists():
        shutil.copy2(path, bundle / path.name)
(bundle / 'COMMIT.txt').write_text(commit + '\n', encoding='utf-8')
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('Download:', archive)
print('Giữ ZIP này làm bằng chứng cấu hình, log, kết quả, biểu đồ và commit mã nguồn.')